First Pipeline to run binding energy


In [ ]:
!uv pip install fairchem-core

In [ ]:
# check through Hugging Face Hub if the package is installed correctly, paste result in chatgpt or claude to confirm
# you mainly need to check off " Read access to contents of all public gated repos you can access" and " Read access to contents of all private gated repos you can access" in tokens settings in Hugging Face Hub, and that the token is correctly set up in your environment variables
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '69d423a991dfba6b651bddfe', 'name': 'umami393', 'fullname': 'Margaret', 'isPro': False, 'avatarUrl': '/avatars/eb54fe80196fee8cdf7c5c7ace19b460.svg', 'orgs': [{'type': 'org', 'id': '64374111a701a7e744c02b0e', 'name': 'princetonu', 'fullname': 'Princeton University', 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/68e396f2b5bb631e9b2fac9a/b3xXusq8Zz3ej8Z6fRTSZ.png'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'readdd', 'role': 'fineGrained', 'createdAt': '2026-04-06T21:59:28.712Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '67f5bb06b8e3712b3da49bfc', 'type': 'model', 'name': 'facebook/UMA'}, 'permissions': ['repo.content.read']}, {'entity': {'_id': '69d423a991dfba6b651bddfe', 'type': 'user', 'name': 'umami393'}, 'permissions': ['repo.content.read']}]}}}}


In [4]:
from fairchem.core import pretrained_mlip

predictor = pretrained_mlip.get_predict_unit(
    model_name="uma-s-1p1",
    device="cpu"
)

W0406 18:04:41.997000 34800 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


checkpoints/uma-s-1p1.pt:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

c:\Users\miaom\miniconda3\envs\cms\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\miaom\.cache\fairchem\models--facebook--UMA. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


iso_atom_elem_refs.yaml: 0.00B [00:00, ?B/s]

In [6]:
# test with calculator 
from ase.build import bulk
from fairchem.core import pretrained_mlip, FAIRChemCalculator

device = "cpu"

atoms = bulk("Si")

model_name = "uma-s-1p1"
predictor = pretrained_mlip.get_predict_unit(model_name)
calc = FAIRChemCalculator(predictor, task_name="omat")

atoms.calc = calc
e = atoms.get_potential_energy()
print(e)

-10.82311692799609


In [8]:
# intialize MOF 
from ase.io import read, write
mof_74 = read("mg_mof74.cif")
mof_74.calc = calc
E_mof_74 = mof_74.get_potential_energy()
print(f"E(Mg-MOF-74) = {E_mof_74:.6f} eV")

E(Mg-MOF-74) = -1171.728085 eV


In [ ]:
from ase.visualize import view
view(mof_74)

In [9]:
from ase.build import molecule
# Optimize isolated CO2
co2 = molecule("CO2")
co2.calc = calc

E_co2 = co2.get_potential_energy()
print(f"E(CO2) = {E_co2:.6f} eV")

E(CO2) = -22.596612 eV


In [ ]:
# to do later, set up initial placements of Co_2 in MOF, then optimize the combined system
# Optimize MOF + CO2
mof_co2 = read("mof_co2_initial.cif")
mof_co2.calc = calc



E_mof_co2 = mof_co2.get_potential_energy()
print(f"E(CO2@Mg-MOF-74) = {E_mof_co2:.6f} eV")


write("mof_co2_relaxed.cif", mof_co2)


# Adsorption energy
delta_E = E_mof_co2 - (E_mof + E_co2)
print(f"Adsorption energy: {delta_E:.4f} eV")


# Mg–CO2 distance
mg_indices = [i for i, atom in enumerate(mof_co2) if atom.symbol == "Mg"]
co2_indices = list(range(len(mof_co2)-3, len(mof_co2)))


for mg_i in mg_indices:
   for co2_j in co2_indices:
       d = mof_co2.get_distance(mg_i, co2_j, mic=True)
       if d < 3.5:
           print(f"Mg[{mg_i}] — {mof_co2[co2_j].symbol}[{co2_j}]: {d:.3f} Å")